# Remove Unused TPS Table Definitions

This notebook removes table definitions that are not referenced by any visible sequence event slot. It edits only an off-software `.tps` file model and writes a new sequence file.

The TPS cleanup itself is offline and does not edit TNT containers. This notebook also contains a separate, active TNMR connection cell that calls `control.connect()` and `control.open(INPUT_TPS_PATH)`. Skip that cell for offline-only use; running all cells requires TNMR COM access.

## 1. Setup

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Find the Hardware Capture repo root from a notebook or script working directory."""

    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        redstone_dir = candidate / "hardware_capture" / "rf" / "redstone"
        if (redstone_dir / "sequence.py").exists() and (redstone_dir / "waveform.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the Hardware Capture repo root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REDSTONE_DIR = REPO_ROOT / "hardware_capture" / "rf" / "redstone"
EXAMPLES_DIR = REDSTONE_DIR / "examples"
OUTPUT_DIR = EXAMPLES_DIR / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)

from hardware_capture.rf.redstone import Sequence

## 2. Select Input And Output Files

Use a copied or exported `.tps` sequence as input. The vendored `1pulse` example already has only used tables, so it removes zero tables; point `INPUT_TPS_PATH` at a messy exported sequence to test cleanup.

In [ ]:
INPUT_TPS_PATH = (
    REDSTONE_DIR
    / "references"
    / "examples"
    / "tnmr-sequences"
    / "common"
    / "1pulse - f1.tps"
)
# Set INPUT_TPS_PATH to your own exported TPS file to inspect another sequence.

OUTPUT_TPS_PATH = OUTPUT_DIR / f"{INPUT_TPS_PATH.stem}_used_tables_only.tps"
OUTPUT_JSON_PATH = OUTPUT_DIR / f"{INPUT_TPS_PATH.stem}_used_tables_only.tps.json"

assert INPUT_TPS_PATH.exists(), INPUT_TPS_PATH
INPUT_TPS_PATH, OUTPUT_TPS_PATH

## 3. Load And Inspect Table Usage

In [ ]:
sequence = Sequence.load(INPUT_TPS_PATH)
table_names_before = [table["name"] for table in sequence.tables if table.get("name")]
referenced_names = sorted(
    {
        reference.table_name
        for reference in sequence.table_references()
        if reference.table_name
    }
)

{
    "table_count_before": len(table_names_before),
    "tables_before": table_names_before,
    "referenced_table_count": len(referenced_names),
    "referenced_tables": referenced_names,
}

In [ ]:
from pathlib import Path
import importlib
import sys

redstone_dir = next(p / "hardware_capture" / "rf" / "redstone" for p in (Path.cwd(), *Path.cwd().parents) if (p / "hardware_capture" / "rf" / "redstone").is_dir())
if str(redstone_dir) not in sys.path:
    sys.path.insert(0, str(redstone_dir))

import tnmrcontrol
importlib.reload(tnmrcontrol)

from tnmrcontrol import TNMRControl
control = TNMRControl()
control.connect()
control.open(INPUT_TPS_PATH)

print("Opened:", control.path)
print("Application:", control.application_info().get("application_object"))

## 4. Remove Unused Table Records

In [ ]:
result = sequence.remove_unused_tables()
result

## 5. Save The Cleaned Sequence

In [ ]:
sequence.write_tps(OUTPUT_TPS_PATH)
sequence.write_json(OUTPUT_JSON_PATH, include_raw_bytes=True)
print("Wrote TPS:", OUTPUT_TPS_PATH)
print("Wrote JSON:", OUTPUT_JSON_PATH)

## 6. Reload And Verify

The cleaned file must still contain definitions for every table referenced by sequence events.

In [ ]:
reloaded = Sequence.load(OUTPUT_TPS_PATH)
reloaded_table_names = {
    table["name"] for table in reloaded.tables if table.get("name")
}
missing = {
    reference.table_name
    for reference in reloaded.table_references()
    if reference.table_name and reference.table_name not in reloaded_table_names
}
assert not missing, sorted(missing)

{
    "reloaded_table_count": len(reloaded_table_names),
    "missing_referenced_tables": sorted(missing),
}